In [59]:
import pandas as pd
import numpy as np
import gspread

from google.oauth2.service_account import Credentials

In [60]:
# Read data from gsheet

SERVICE_ACCOUNT_FILE = '../../key/credentials.json'
SCOPES = ['https://www.googleapis.com/auth/spreadsheets',
          'https://www.googleapis.com/auth/drive']

creds = Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)
client = gspread.authorize(creds)

sheet = client.open('[4] AI QC Inbound CRM Review 语音智能质检打标复审').worksheet('New 5')
data = sheet.get_all_records()

df = pd.DataFrame(data[1:], columns=data[0])
df.to_csv('../../raw_data/new_hotline_4.csv', index=False)

In [61]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5106 entries, 0 to 5105
Data columns (total 23 columns):
 #   Column                            Non-Null Count  Dtype 
---  ------                            --------------  ----- 
 0   No                                5106 non-null   int64 
 1   Tanggal Pengerjaan                5106 non-null   object
 2   Checker                           5106 non-null   object
 3   ASI/AFI                           5106 non-null   object
 4   Waktu Inbound                     5106 non-null   object
 5   Nomor Inbound                     5106 non-null   object
 6   Nama Agent                        5106 non-null   object
 7   Call ID                           5106 non-null   object
 8   Durasi Bicara                     5106 non-null   object
 9   Detik                             5106 non-null   object
 10  Total Tiket Riskan                5106 non-null   object
 11  Apakah Riskan?                    5106 non-null   object
 12  Tinjauan CS         

In [62]:
# Lowering all data and removing extra spaces
df_clean = (
    df.copy()
    .drop(columns=['No'])
    .apply(lambda x: x.str.lower() if x.dtype == 'object' else x)
    .apply(lambda x: x.str.strip() if x.dtype == 'object' else x)
)

# lowering all columns names and removing extra spaces
df_clean.columns = (
    df_clean
    .columns.str.lower().str.strip()
    .str.replace(' ', '_')
)

# data types
df_clean['tanggal_pengerjaan'] = pd.to_datetime(df_clean['tanggal_pengerjaan'], errors='coerce')

df_clean

,tanggal_pengerjaan,checker,asi/afi,waktu_inbound,nomor_inbound,nama_agent,call_id,durasi_bicara,detik,total_tiket_riskan,...,status,sampling_user_side,hasil_pemeriksaan_kualitas_(old),hasil_asr,hasil_pemeriksaan_kualitas,efektif,kejelasan_suara,suara_lain,kelengkapan_rekaman,agent_sampling
0,2026-03-16,neneng,asi,2026-03-15 20:00:46,0878****0038,isnaini,a84d3724f72046409ec47e2a82cee184,0:07:58,2026-03-15 20:01:40,,...,,done,,,percakapan normal,miss target/ not hc,"sangat jelas, tidak bising sama sekali",0 satu pembicara,0 utuh,
1,2026-03-16,neneng,asi,2026-03-15 20:00:46,0878****0038,isnaini,a84d3724f72046409ec47e2a82cee184,0:07:58,2026-03-15 20:01:47,,...,,done,,,percakapan normal,miss target/ not hc,cukup jelas,0 satu pembicara,0 utuh,
2,2026-03-16,neneng,asi,2026-03-15 20:00:46,0878****0038,isnaini,a84d3724f72046409ec47e2a82cee184,0:07:58,2026-03-15 20:01:50,,...,,done,,,percakapan normal,miss target/ not hc,cukup jelas,0 satu pembicara,0 utuh,
3,2026-03-16,neneng,asi,2026-03-15 20:00:46,0878****0038,isnaini,a84d3724f72046409ec47e2a82cee184,0:07:58,2026-03-15 20:01:52,,...,,done,,,percakapan normal,miss target/ not hc,cukup jelas,0 satu pembicara,0 utuh,
4,2026-03-16,neneng,asi,2026-03-15 20:00:46,0878****0038,isnaini,a84d3724f72046409ec47e2a82cee184,0:07:58,2026-03-15 20:01:57,,...,,done,,,percakapan normal,miss target/ not hc,"sangat jelas, tidak bising sama sekali",0 satu pembicara,0 utuh,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5101,2026-03-31,reza,afi,2026-03-12 15:16:38,0812****8867,hidayati,cc14e215aef149a6939f57236829df2d,0:08:53,2026-03-12 15:23:56,,...,,done,,,percakapan normal,miss target/ not hc,"sangat jelas, tidak bising sama sekali",0 satu pembicara,1 tidak utuh,
5102,2026-03-31,reza,afi,2026-03-12 15:16:38,0812****8867,hidayati,cc14e215aef149a6939f57236829df2d,0:08:53,2026-03-12 15:23:58,,...,,done,,,percakapan normal,miss target/ not hc,cukup jelas,0 satu pembicara,0 utuh,
5103,2026-03-31,reza,afi,2026-03-12 15:16:38,0812****8867,hidayati,cc14e215aef149a6939f57236829df2d,0:08:53,2026-03-12 15:24:02,,...,,done,,,percakapan normal,miss target/ not hc,"sangat jelas, tidak bising sama sekali",0 satu pembicara,0 utuh,
5104,2026-03-31,reza,afi,2026-03-12 15:16:38,0812****8867,hidayati,cc14e215aef149a6939f57236829df2d,0:08:53,2026-03-12 15:24:12,,...,,done,,,percakapan normal,miss target/ not hc,sulit didengar,0 satu pembicara,0 utuh,


In [71]:
# rata-rata pengerjaan per hari
count = df_clean['tanggal_pengerjaan'].value_counts().sort_index()
count

tanggal_pengerjaan
2026-03-16    335
2026-03-17    370
2026-03-25    642
2026-03-26    912
2026-03-27    965
2026-03-30    907
2026-03-31    975
Name: count, dtype: int64

In [64]:
count = pd.DataFrame(count)
round(count['count'].mean())

729

In [66]:
# cumulative sum for total data

df_clean.groupby('tanggal_pengerjaan')['call_id'].size().sort_index().cumsum()


tanggal_pengerjaan
2026-03-16     335
2026-03-17     705
2026-03-25    1347
2026-03-26    2259
2026-03-27    3224
2026-03-30    4131
2026-03-31    5106
Name: call_id, dtype: int64

In [67]:
df_clean[df_clean['asi/afi'] == 'afi'].groupby('tanggal_pengerjaan')['call_id'].size().sort_index().cumsum()

tanggal_pengerjaan
2026-03-26    411
2026-03-27    660
2026-03-31    751
Name: call_id, dtype: int64

In [68]:
# cumulative sum for unique call_id
df_clean.groupby('tanggal_pengerjaan')['call_id'].nunique().sort_index().cumsum()

tanggal_pengerjaan
2026-03-16     5
2026-03-17    10
2026-03-25    15
2026-03-26    24
2026-03-27    34
2026-03-30    43
2026-03-31    53
Name: call_id, dtype: int64

In [69]:
df_clean[df_clean['asi/afi'] == 'afi'].groupby('tanggal_pengerjaan')['call_id'].nunique().sort_index().cumsum()

tanggal_pengerjaan
2026-03-26    4
2026-03-27    7
2026-03-31    8
Name: call_id, dtype: int64